# Data Wrangling Techniques

A practical walkthrough of data standardization, categorical encoding, missing-value handling, transformations, and feature engineering using the Stack Overflow Developer Survey.

**Goal:** prepare survey data for analysis while documenting decisions that can affect interpretation.

## 1. Load and briefly inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

print("Shape:", df.shape)
display(df.head())

## 2. Standardize inconsistent categorical values

Inspect the distinct values first. Mapping should be based on values actually present in the dataset; unknown variants should not be silently changed.

In [ ]:
print("Unique countries:", df["Country"].nunique())
display(df["Country"].value_counts().head(15))

In [ ]:
country_map = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "Russian Federation": "Russia",
    "Iran, Islamic Republic of...": "Iran",
    "Hong Kong (S.A.R.)": "Hong Kong",
    "Republic of Korea": "South Korea",
}

df["Country"] = df["Country"].replace(country_map)
df["Country"] = df["Country"].str.strip()

edlevel_map = {
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": "Bachelor’s degree",
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master’s degree",
    "Primary/elementary school": "Primary school",
}
df["EdLevel"] = df["EdLevel"].replace(edlevel_map)

display(df["Country"].value_counts().head(10))

## 3. Encode categorical variables

`Employment` can contain multiple selections separated by semicolons. Split the responses and create one indicator column per employment category instead of treating each combination as a single category.

The encoded columns are Boolean indicators. The original `Employment` column is retained.

In [ ]:
employment_dummies = df["Employment"].str.get_dummies(sep=";")
employment_dummies = employment_dummies.add_prefix("Employment_")

df = pd.concat([df, employment_dummies], axis=1)
display(employment_dummies.head())
print("Encoded employment categories:", employment_dummies.shape[1])

## 4. Review and handle missing values

First, identify the columns with the most missing observations. Missingness is not automatically an error; it may mean the respondent did not answer a question or the question was not applicable.

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
display(missing_counts.head(10))

### 4.1 Numerical values: compensation

Median imputation is demonstrated below, but it should not be applied blindly. Compensation is highly skewed, and filling every unanswered salary with the median creates a large artificial cluster and changes the distribution.

We preserve the original `ConvertedCompYearly` values and create a separate imputed column only to demonstrate the method. Use the original reported values for distribution analysis.

In [ ]:
compensation = df["ConvertedCompYearly"]
compensation_median = compensation.median()

df["ConvertedCompYearly_MedianImputed"] = compensation.fillna(compensation_median)

print("Original missing values:", compensation.isna().sum())
print("Median used:", compensation_median)
print("Missing after demonstration imputation:",
      df["ConvertedCompYearly_MedianImputed"].isna().sum())

### 4.2 Categorical values: RemoteWork

For a categorical field, mode imputation replaces missing entries with the most frequent response. Keep the original column if you need to compare or audit the change.

In [ ]:
remote_work_mode = df["RemoteWork"].mode(dropna=True).iloc[0]
df["RemoteWork_ModeImputed"] = df["RemoteWork"].fillna(remote_work_mode)

print("Mode:", remote_work_mode)
print("Original missing values:", df["RemoteWork"].isna().sum())
print("Missing after imputation:", df["RemoteWork_ModeImputed"].isna().sum())

## 5. Feature scaling and transformation

Apply Min-Max scaling and a log transformation to **reported compensation only**. Missing compensation remains missing; it is not replaced with a fabricated value for these analyses.

Min-Max scaling is sensitive to extreme values. The survey contains very high compensation outliers, so most reported values may appear compressed near zero.

In [ ]:
reported_comp = df["ConvertedCompYearly"]
valid_comp = reported_comp.dropna()

comp_min = valid_comp.min()
comp_max = valid_comp.max()

df["ConvertedCompYearly_MinMax"] = (
    (reported_comp - comp_min) / (comp_max - comp_min)
    if comp_max != comp_min else np.nan
)

# log1p handles zero compensation values and leaves missing values as missing
df["ConvertedCompYearly_Log"] = np.log1p(reported_comp)

print("Reported compensation rows:", valid_comp.shape[0])
print("Missing values retained:", df["ConvertedCompYearly_MinMax"].isna().sum())
print("Skew before:", valid_comp.skew())
print("Skew after log1p:", df["ConvertedCompYearly_Log"].dropna().skew())
display(df[["ConvertedCompYearly", "ConvertedCompYearly_MinMax",
            "ConvertedCompYearly_Log"]].describe())

### Visual comparison

The histograms below use respondents who reported compensation. The log scale makes the shape easier to inspect, but it does not remove the need to investigate outliers or choose an appropriate analysis method.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].hist(valid_comp, bins=50)
axes[0].set_title("Reported compensation")
axes[0].set_xlabel("Annual compensation")

axes[1].hist(df["ConvertedCompYearly_MinMax"].dropna(), bins=50)
axes[1].set_title("Min-Max scaled")
axes[1].set_xlabel("Scaled value")

axes[2].hist(df["ConvertedCompYearly_Log"].dropna(), bins=50)
axes[2].set_title("Log1p transformed")
axes[2].set_xlabel("log(1 + annual compensation)")

plt.tight_layout()
plt.show()

## 6. Feature engineering: experience bands

`YearsCodePro` includes text responses. Convert the two text categories to numeric approximations, then create mutually exclusive experience bands with clearly defined boundaries:

- Junior: 0–2 years
- Mid: more than 2 through 5 years
- Senior: more than 5 through 10 years
- Expert: more than 10 years

The approximations (`0.5` and `51`) are assumptions for grouping, not exact reported values.

In [ ]:
years = df["YearsCodePro"].replace({
    "Less than 1 year": 0.5,
    "More than 50 years": 51
})
years = pd.to_numeric(years, errors="coerce")

df["YearsCodePro_numeric"] = years
df["ExperienceLevel"] = pd.cut(
    years,
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=["Junior (0–2)", "Mid (>2–5)", "Senior (>5–10)", "Expert (>10)"],
    right=True
)

display(df["ExperienceLevel"].value_counts(dropna=False).sort_index())

## 7. Summary

In this notebook, we:
- standardized selected categorical labels;
- encoded a multi-select employment field into separate indicators;
- compared numerical and categorical missing-value imputation;
- scaled and log-transformed reported compensation while preserving missing values;
- engineered experience bands from a text-based field.

**Key takeaway:** data-wrangling choices affect the distribution and interpretation of results. Keep original fields when transformations or imputations need to be audited.